In [ ]:
import sys
import logging
sys.path.insert(0, "..")   # make validation/src importable from notebook

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s  %(message)s",
)

from build_amy_epw import AMYEPWBuilder

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")  # loads validation/.env

NREL_API_KEY = os.environ["NREL_API_KEY"]
NREL_EMAIL   = os.environ["NREL_EMAIL"]

STATION_ID = "726810-24131"
YEAR       = 2023
OUT_PATH   = f"test_{STATION_ID.replace('-','_')}_{YEAR}.epw"

In [ ]:
# --- Step 1: station metadata only (cheap, no API key needed) ---
builder = AMYEPWBuilder(STATION_ID, YEAR, NREL_API_KEY, NREL_EMAIL)
meta = builder.fetch_station_metadata()
print(meta)

In [ ]:
# --- Step 2: fetch + parse ISD met data ---
met = builder.fetch_met()
print(f"Shape: {met.shape}")
print(f"Date range: {met.index[0]}  →  {met.index[-1]}")
print(f"\nMissing values per column:")
print(met.isna().sum())
met.head()

In [ ]:
# --- Step 3: quick sanity plots for met data ---
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
met["temp_C"].plot(ax=axes[0], title="Dry-bulb temperature (°C)")
met["wind_speed_ms"].plot(ax=axes[1], title="Wind speed (m/s)")
met["rh_pct"].plot(ax=axes[2], title="Relative humidity (%)")
plt.tight_layout()
plt.show()

In [ ]:
# --- Step 4: fetch solar (requires valid NREL API key + email) ---
solar = builder.fetch_solar()
print(f"Shape: {solar.shape}")
print(solar.describe().round(1))
solar.head()

In [ ]:
# --- Step 5: full build → write EPW ---
epw_path = builder.build(OUT_PATH)
print(f"Written: {epw_path}")

In [ ]:
# --- Step 6: verify EPW round-trip via ladybug ---
from ladybug.epw import EPW
import numpy as np

epw = EPW(epw_path)
print(f"Location : {epw.location.city}, {epw.location.state}, {epw.location.country}")
print(f"Lat/Lon  : {epw.location.latitude:.4f}, {epw.location.longitude:.4f}")
print(f"Elevation: {epw.location.elevation} m")
print(f"Timezone : UTC{int(epw.location.time_zone):+d}")

dbt = epw.dry_bulb_temperature.values
print(f"\nDry-bulb  min={np.min(dbt):.1f}°C  mean={np.mean(dbt):.1f}°C  max={np.max(dbt):.1f}°C")
dni = epw.direct_normal_radiation.values
print(f"DNI       min={np.min(dni):.0f}  mean={np.mean(dni):.0f}  max={np.max(dni):.0f}  Wh/m²")